[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/EelcoHoogendoorn/numga?ref=rewrite)

# Multi-View Reconstruction & Bundle Adjustment in PGA2D

Multi-view geometry and bundle adjustment provide a compelling showcase for **extensors as active geometric entities** in a real-world setting. Rather than treating transformations as abstract matrix arrays, extensors operate directly on typed geometric subspaces with clear physical intuition:

* **Inverse-Perspective Cone Lifting**: Sensor pixel detections are lifted back through projective camera extensors into perspective quadric cones, turning 1D transverse sensor precision into full 2D directional constraints.
* **Spatial Transport via Motors**: Rigid camera motions (motors in SE(2)) seamlessly transport quadric cones between camera local coordinates and world space via sandwiching (`motors >> cones(motors << Point)`).
* **Additive Projective Constraints**: Because quadrics represent quadratic costs, directional constraints from convergent viewpoints combine by direct addition, fusing multiple camera cones into localized Gaussian precision ellipsoids (splats).
* **Polar Duality of Center and Infinity**: By dual projective polarity, the Euclidean center of a quadric is the pole of the ideal plane at infinity `w`. Solving `splats(center) = w` extracts the reconstructed point in closed form without ray heuristics.
* **Active Inverses and Spectral Operators**: Inverting, regularizing, and evaluating eigenvalues of extensor operators directly yields geometric covariances and uncertainty ellipsoids.
* **Motor Optimization & Covariance Extensors**: Camera poses are optimized along the Lie algebra via bivector commutators (adjoint extensors), producing posterior pose covariance operators directly from the reduced stiffness Hessian.

---

### Pipeline Overview
1. **Scene Setup**: 6 scene points and a convergent 3-camera rig.
2. **Perspective Cones**: Projective camera extensors and pullback of transverse sensor uncertainty.
3. **Triangulation**: Quadric fusion via the pole of the plane at infinity `w`, and pre-optimization plot.
4. **Alternating Bundle Adjustment**: Closed-form triangulation alternating with Lie-algebra camera pose updates.
5. **Coupled Bundle Adjustment**: Schur-complement point elimination, Sampson depth reweighting, and pose covariance.

In [ ]:
# Imports: plotting, numerical backend, and PGA2D multivector types
import sys
from pathlib import Path

# Ensure project root is in sys.path when running from notebook directory:
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "examples").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

from IPython.display import Image, display
import matplotlib.pyplot as plt
import numpy as np

from examples.geometry.multiview import render
from examples.geometry.multiview.types import (
    Camera,      # Projective transformation extensor: Point -> Point
    Motor,       # Even-grade multivector: rigid planar motions in SE(2)
    Plane,       # Grade-1 multivector: lines in 2D / planes in 3D (a*x + b*y + c*w = 0)
    Point,       # Grade-2 multivector: 2D points [x, y, w]
    PointMap,    # Projective camera mapping extensor: Point -> Point
    Quadric,     # Bilinear form extensor: Plane <- Point (or Point <- Plane)
    Scalar,      # Grade-0 multivector: scalar values
    Twist,       # Lie algebra se(2) generator for infinitesimal twists
    TwistMap,    # Lie algebra operator extensor: Twist -> Twist
    coordinates, # Readout helper: converts normalized Point multivectors to NumPy arrays
    mv,          # Basis multivector generator (mv.x, mv.y, mv.w, mv.xy, mv.xw, mv.yw)
    point,       # Algebraic constructor: converts NumPy arrays to typed Point multivectors
    stack,       # Extensor stacking utility across axes
    w,           # The ideal line/plane at infinity: mv.w
)


## 1. Setup Scene and 3-Camera Rig

We place 6 ground truth scene points in front of a convergent 3-camera rig with a 1.5 m baseline (left, center, and right viewpoints). The projective camera model joins scene points with the pinhole origin and meets them with the sensor plane `y = 1`.

In [ ]:
# 1. Define 6 ground truth scene points in front of the cameras:
coords = np.array([
    [ 0.15, 0.85],
    [-0.43, 1.15],
    [ 0.50, 1.50],
    [ 0.03, 1.85],
    [ 0.65, 2.20],
    [-0.60, 2.55],
])
true_points = point(coords)                         # [n_points] Point

# 2. Construct 3 convergent camera poses (left, center, right):
xs = np.array([-0.75, 0.0, 0.75])
thetas = np.radians([18.0, 0.0, -18.0])
true_poses = ((mv.xw * xs) * 0.5).exp() * ((mv.xy * thetas) * 0.5).exp()  # [n_cams] Motor

# 3. Construct projective camera model (Point -> Point):
c0 = point([0.0, 0.0])                              # Point (pinhole origin)
screen = mv.y - mv.w                                # Line (sensor line y = 1)
projection = (c0 & Point) ^ screen                  # PointMap: Point -> Point

print(f"Cameras: {len(xs)} viewpoints, baseline: {np.ptp(xs):.2f} m")
print(f"Scene: {len(coords)} points spanning depths {coords[:, 1].min():.2f} m to {coords[:, 1].max():.2f} m")

# Plot 1: Setup - Ground truth scene points and convergent camera rig
fig = render.draw_top_down_figure(
    poses=true_poses,
    points=true_points,
    cam_colors=["#0284c7", "#8b5cf6", "#ec4899"],
)
plt.title("1. Setup: Ground Truth Scene & Convergent Camera Rig")
plt.show()

## 2. Sensor Pixel Measurements & Perspective Sight Cones

Projecting scene points produces measured sensor pixels. At each pixel, transverse uncertainty is a rank-1 precision dyad on the sensor plane. Pulling these dyads back through the camera projection map lifts them into 2D perspective cone quadrics whose uncertainty naturally widens with depth.

In [ ]:
# 1. Project true scene points into local camera frames and extract sensor pixels:
local_points = true_poses << true_points[:, None]   # [n_points, n_cams] Point
projs = projection(local_points)                    # [n_points, n_cams] Point
pixels = projs / (mv.w & projs)                     # [n_points, n_cams] Point (homogeneous coordinates)

# 2. Per-pixel transverse precision dyads on the sensor plane:
# At each measured pixel, transverse deviation is penalized quadratically as (normal & Point)^2.
normal = mv.x - mv.w * (mv.x & pixels)              # [n_points, n_cams] Plane (transverse normal lines)
sensor_discs = normal * (normal & Point)            # [n_points, n_cams] Plane <- Point (sensor precision dyads)

# 3. Inverse-perspective transform: pull sensor discs back into perspective cones:
pullback = projection.transpose()(Plane.dual()).dual_inverse()  # Plane <- Plane (adjoint C^T)
cones = pullback(sensor_discs(projection))          # [n_points, n_cams] Quadric (Plane <- Point)

print("Perspective sight cones shape:", cones.shape)
print("Pixel coordinates on left camera sensor:", coordinates(pixels[:, 0])[:, 0])
print("Pixel coordinates on right camera sensor:", coordinates(pixels[:, 2])[:, 0])

# Plot 2: Adding Cones - Perspective sight rays fanning out through scene points
world_cones = true_poses >> cones(true_poses << Point)  # [n_points, n_cams] Quadric in world frame
fig = render.draw_top_down_figure(
    poses=true_poses,
    cones=world_cones,
    points=true_points,
    cam_colors=["#0284c7", "#8b5cf6", "#ec4899"],
)
plt.title("2. Adding Cones: Perspective Sight Rays from All Cameras")
plt.show()

## 3. Closed-Form Triangulation & Pre-Optimization Geometry

Camera poses transport local sight cones to the world frame via double-sided motor transport. Summing sight cones across cameras fuses directional constraints into Gaussian precision ellipsoids (splats), whose centers are extracted in closed form as poles of infinity. Below, we perturb Camera 2 to introduce pose error and triangulate the initial splats.

In [ ]:
def triangulate_cones(poses: Motor, cones: Quadric) -> tuple[Point, Quadric]:
    """Triangulate scene points as poles of infinity from fused sight cone quadrics."""
    world_cones = poses >> cones(poses << Point)    # [n_points, n_cams] Quadric
    splats = world_cones.sum(axis=-1)               # [n_points] Quadric (fused splats)
    points = (splats + mv.w * (mv.w & Point)).solve(mv.w).normalized()  # [n_points] Point
    return points, splats


# Perturb Camera 2 (right camera): +8 cm x, -4 cm y, -4 deg yaw:
pert2 = ((mv.xw * 0.08 - mv.yw * 0.04) * 0.5).exp() * ((-mv.xy * np.radians(4.0)) * 0.5).exp()
initial_poses = stack([true_poses[0], true_poses[1], true_poses[2] * pert2])  # [n_cams] Motor

# Initial closed-form triangulation:
init_points, init_splats = triangulate_cones(initial_poses, cones)
init_rmse = np.sqrt(np.mean((coordinates(init_points) - coords) ** 2))
print(f"Pre-optimization point RMSE (Cam 2 perturbed): {init_rmse:.6f} m ({init_rmse * 100:.2f} cm)")

# Plot 3: Adding Quads - Fused precision splats under perturbed pose
init_world_cones = initial_poses >> cones(initial_poses << Point)
fig = render.draw_top_down_figure(
    poses=initial_poses,
    cones=init_world_cones,
    splats=init_splats,
    points=init_points,
    cam_colors=["#0284c7", "#8b5cf6", "#ec4899"],
)
plt.title("3. Adding Quads: Fused Precision Splats under Perturbed Pose")
plt.show()

## 4. Alternating Bundle Adjustment

We alternate between closed-form triangulation and Lie-algebra camera pose updates:
1. **Triangulate Points**: Closed-form pole-of-infinity solve `triangulate_cones(poses, cones)`.
2. **Evaluate Polar Plane Residuals**: Map points to local frames and evaluate cones: `res = cones(local_points)`.
3. **Lie-Algebra Jacobians**: Pose variations under se(2) twist commutators yield `j = -cones(Twist.commutator(local_points))`.
4. **Accumulate & Solve**: Form `h = (j.T @ j).sum()` and `rhs = -(j.T @ res).sum()`, solve for twist step, and update poses with `poses = poses * (step * 0.5).exp()`.

In [ ]:
# Alternating Gauss-Newton optimization:
poses = initial_poses                                        # [n_cams] Motor
damping = 0.5
iterations = 8

history = []

print(f"{'Iter':<5} | {'Point RMSE (m)':<16} | {'Step Norm':<14}")
print("-" * 42)

for it in range(iterations):
    points, splats = triangulate_cones(poses, cones)
    history.append((poses, points, splats))

    local_points = poses << points[:, None]                 # [n_points, n_cams] Point
    res = cones(local_points)                               # [n_points, n_cams] Line (polar residuals)
    j = -cones(Twist.commutator(local_points))              # [n_points, n_cams] Line <- Twist

    h = (j.transpose()(j)).sum(axis=0)                      # [n_cams] Twist <- Twist
    rhs = -(j.transpose()(res)).sum(axis=0)                 # [n_cams] Twist

    step = h.solve(rhs)                                     # [n_cams] Twist
    step = step.at[0].set(step[0] * 0)                      # Anchor reference camera 0
    poses = poses * (step * (0.5 * damping)).exp()          # [n_cams] Motor

    current_rmse = np.sqrt(np.mean((coordinates(points) - coords) ** 2))
    print(f"Iteration {it + 1:2d} | Point RMSE: {current_rmse:.6f} m | Step norm: {np.linalg.norm(step.kernel):.4e}")

opt_poses_alt = poses                                       # [n_cams] Motor
opt_points_alt, opt_splats_alt = triangulate_cones(opt_poses_alt, cones)
history.append((opt_poses_alt, opt_points_alt, opt_splats_alt))

alt_rmse = np.sqrt(np.mean((coordinates(opt_points_alt) - coords) ** 2))
print(f"\nAlternating Loop Final Point RMSE: {alt_rmse:.6f} m ({alt_rmse * 100:.2f} cm)")

# Plot 4: Convergence Animation of Alternating Bundle Adjustment
gif_path = render.animate_top_down_convergence(
    history=history,
    local_cones=cones,
    cam_colors=["#0284c7", "#8b5cf6", "#ec4899"],
    gif_path=Path("examples/plots/multiview_convergence.gif"),
    fps=3,
    auto_increment=False,
)
if gif_path and Path(gif_path).exists():
    display(Image(filename=str(gif_path)))

## 5. Posterior Pose Covariance & Uncertainty Geometry

Inverting the accumulated Gauss-Newton Hessian directly yields the posterior pose covariance operator on the twist Lie algebra. We extract the 1-sigma uncertainty ellipses for camera centers and the angular uncertainty fans.

In [ ]:
# Extract pose covariance and render final state:
h_final = (j.transpose()(j)).sum(axis=0)
pose_covariance = h_final.pinv(rcond=1e-4)

opt_world_cones_final = opt_poses_alt >> cones(opt_poses_alt << Point)
fig = render.draw_top_down_figure(
    poses=opt_poses_alt,
    cones=opt_world_cones_final,
    splats=opt_splats_alt,
    points=opt_points_alt,
    pose_covariances=pose_covariance,
    cam_colors=["#0284c7", "#8b5cf6", "#ec4899"],
)
plt.title("5. Final State: Precision Splats & Pose Covariance")
plt.show()

print("\n=== Reconstruction Accuracy Progression ===")
print(f"1. Pre-optimization (perturbed Cam 2): {init_rmse * 100:.3f} cm")
print(f"2. Converged Landmark RMSE           : {alt_rmse * 100:.3f} cm")

## 6. Key Takeaways & Summary

1. **Simple to Complex**:
   * **Triangulation**: Summing raw perspective sight cones extracts points as quadric centers (`center: Point`, the pole of infinity `w`).
   * **Alternating Loop**: Alternating between closed-form triangulation and camera twist updates needs only 6 lines of math.
   * **Covariance**: Inverting the accumulated Hessian yields posterior pose covariance operators.
2. **Unified Geometric Subspaces**:
   Camera poses are motors, optical projection is a collineation, sight observations are quadrics, and pose variations are bivector commutators.
3. **Gaussian Splat Geometry**:
   The fused quadric `splats` provides instantaneous directional uncertainty ellipsoids for downstream filtering and rendering.